# D5 - Decorators, Context Managers & Caching

## Objective
Demonstrate meta-programming concepts in Python: custom function decorators (`@timeit`, `@retry`), context managers (`__enter__`, `__exit__`, `@contextmanager`), and function response caching using `functools.lru_cache`.

## Concepts Covered
- **Custom Decorators & `functools.wraps`**: Preserving function signature metadata when augmenting behavior.
- **Parameterized Decorators (`@retry`)**: Passing `max_attempts` argument to control retry logic.
- **Context Managers (`__enter__`, `__exit__`, `@contextmanager`)**: Guaranteeing resource cleanup and scope setup.
- **Function Caching (`lru_cache`)**: Caching expensive config lookups and monitoring cache stats (`cache_info()`, `cache_clear()`).
- **Pipeline Integration**: Integrating execution timing and retries into task processing stages.

## Project Implementation
The utility implementations reside in:
- `app/utils/decorators.py`: `@timeit` for duration logging and `@retry` for resilient execution.
- `app/utils/context_managers.py`: `PipelineResourceContext` class context manager and `@managed_pipeline_file` generator context manager.
- `app/utils/cache.py`: `get_pipeline_step_config` cached function using `lru_cache(maxsize=128)`.

## Demonstration
Below, we demonstrate each decorator, context manager, caching behavior, and pipeline integration.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import time
from app.utils.decorators import timeit, retry

# 1. Custom Decorator Demonstration (@timeit and @retry)
@timeit
@retry(max_attempts=2)
def dummy_computation(x: int) -> int:
    """Computes square of x."""
    return x * x

print("Executing decorated function...")
res = dummy_computation(5)
print(f"Result: {res}")
print(f"Preserved Docstring: '{dummy_computation.__doc__}'")
print(f"Preserved Name: '{dummy_computation.__name__}'")

Executing decorated function...
Result: 25
Preserved Docstring: 'Computes square of x.'
Preserved Name: 'dummy_computation'


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import tempfile
from app.utils.context_managers import PipelineResourceContext, managed_pipeline_file

# 2. Context Manager Demonstration (PipelineResourceContext and managed_pipeline_file)
print("Demonstrating PipelineResourceContext (class-based __enter__/__exit__):")
with PipelineResourceContext("demo_resource") as res:
    print(f"  Resource '{res.resource_name}' acquired? {res.is_acquired}")
print(f"  Resource acquired after exiting block? {res.is_acquired}")

print("\nDemonstrating managed_pipeline_file (@contextmanager generator):")
with tempfile.TemporaryDirectory() as tmp_dir:
    sample_path = Path(tmp_dir) / "sample_managed.txt"
    with managed_pipeline_file(sample_path, mode="w") as f:
        f.write("Managed content via contextmanager decorator")
    print(f"  File created and written: '{sample_path.read_text().strip()}'")

Demonstrating PipelineResourceContext (class-based __enter__/__exit__):
  Resource 'demo_resource' acquired? True
  Resource acquired after exiting block? False

Demonstrating managed_pipeline_file (@contextmanager generator):
  File created and written: 'Managed content via contextmanager decorator'


In [3]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.utils.cache import get_pipeline_step_config

# 3. Caching Demonstration (lru_cache)
get_pipeline_step_config.cache_clear()

print("Testing LRU Caching for get_pipeline_step_config:")
cfg1 = get_pipeline_step_config("TaskValidationStep")
print(f"  First call result: {cfg1}")
print(f"  Cache Info after 1st call: {get_pipeline_step_config.cache_info()}")

cfg2 = get_pipeline_step_config("TaskValidationStep")
print(f"  Second call result (from cache): {cfg2}")
print(f"  Cache Info after 2nd call: {get_pipeline_step_config.cache_info()}")

assert get_pipeline_step_config.cache_info().hits == 1

Testing LRU Caching for get_pipeline_step_config:
  First call result: {'title_required': True, 'max_title_length': 100, 'min_title_length': 1, 'environment': 'production'}
  Cache Info after 1st call: CacheInfo(hits=0, misses=1, maxsize=128, currsize=1)
  Second call result (from cache): {'title_required': True, 'max_title_length': 100, 'min_title_length': 1, 'environment': 'production'}
  Cache Info after 2nd call: CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


## Actual Output
The code cells demonstrate:
1. Duration logging and retry mechanism for decorated functions while preserving `__name__` and `__doc__`.
2. Guaranteed file handle and resource cleanup upon exiting context manager blocks.
3. Successful cache hit (`hits=1`) on repeated invocations of `get_pipeline_step_config`.

## Key Observations
- Using `functools.wraps` is critical to prevent decorators from obscuring function names and docstrings.
- Context managers guarantee resource cleanup even if exceptions occur inside the code block.
- `lru_cache` significantly speeds up repeated read-heavy lookups, but caches must be invalidated if underlying configurations change.

## Conclusion
Decorators, context managers, and caching provide powerful abstractions that enhance code reusability, safety, and performance.